# 🧬 Phase 1: Biomedical KG — Data Download & RAPIDS GPU Preprocessing

**Project:** BioKG Disease Link Predictor  
**Dataset:** DRKG (Drug Repurposing Knowledge Graph) — 5.8M biological triples  
**Tech:** NVIDIA RAPIDS cuDF/cuGraph + DRKG exploration  

---

## 🎓 Phase 1 Objective
1. **What a Knowledge Graph is** — entities, relations, triples
2. **NVIDIA RAPIDS cuDF** — GPU-accelerated DataFrames (loading millions of rows in < 1s)
3. **cuGraph** — Identifying central disease 'hubs' via PageRank

## ⚡ Before Starting
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells

## Step 0: Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "❌ Stop! No GPU found. Go to Runtime -> Change runtime type -> T4 GPU."
print(f"\n✅ GPU Connected: {torch.cuda.get_device_name(0)}")

## Step 1: Install NVIDIA RAPIDS
This takes ~3-5 minutes. ☕

In [ ]:
import torch
import subprocess
cuda_version = torch.version.cuda
major = int(cuda_version.split('.')[0]) if cuda_version else 12
rapids_suffix = 'cu11' if major == 11 else 'cu12'

print(f"📦 Installing RAPIDS for CUDA {cuda_version}...")
!pip install cudf-{rapids_suffix} cugraph-{rapids_suffix} --extra-index-url=https://pypi.nvidia.com -q
print("✅ RAPIDS installed.")

## Step 2: Download & Smart Extraction

> we will automatically find the largest file to avoid picking up 0-byte metadata files.

In [ ]:
import requests
import tarfile
import os
from pathlib import Path
from tqdm.notebook import tqdm

DATA_DIR = Path('data/raw')
if DATA_DIR.exists():
    import shutil
    shutil.rmtree(DATA_DIR) # Clean start
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("📥 Downloading DRKG dataset...")
url = 'https://dgl-data.s3-accelerate.amazonaws.com/dataset/DRKG/drkg.tar.gz'
dest = DATA_DIR / 'drkg.tar.gz'

response = requests.get(url, stream=True)
total = int(response.headers.get('content-length', 0))
with open(dest, 'wb') as f, tqdm(total=total, unit='iB', unit_scale=True) as pb:
    for chunk in response.iter_content(8192):
        pb.update(f.write(chunk))

print("\n📦 Extracting...")
with tarfile.open(dest, 'r:gz') as tar:
    # Extract ALL files
    tar.extractall(path=DATA_DIR)

# 🥇 SMART SELECT: Pick the LARGEST .tsv file (this bypasses hidden macOS metadata files)
tsv_files = list(DATA_DIR.rglob('*.tsv'))
DRKG_TSV = max(tsv_files, key=lambda f: f.stat().st_size)

print(f"\n✅ Target File Found: {DRKG_TSV.name}")
print(f"   Path: {DRKG_TSV}")
print(f"   Size: {DRKG_TSV.stat().st_size / 1024**2:.1f} MB")

## Step 3: GPU vs CPU Benchmark

> **Notice:** We use `encoding='latin-1'` to safely read biological symbols.

In [ ]:
import time
import cudf
import pandas as pd

print('⚡ Starting Benchmark...')

print('\n🐢 Loading with pandas (CPU)...')
t0 = time.time()
pdf = pd.read_csv(str(DRKG_TSV), sep='\t', header=None, 
                  names=['head', 'relation', 'tail'], encoding='latin-1')
cpu_time = time.time() - t0
print(f'   Time: {cpu_time:.2f}s | Rows: {len(pdf):,}')

print('\n⚡ Loading with cuDF (GPU)...')
t0 = time.time()
gdf = cudf.read_csv(str(DRKG_TSV), sep='\t', header=None, 
                   names=['head', 'relation', 'tail'], encoding='latin-1')
gpu_time = time.time() - t0
print(f'   Time: {gpu_time:.2f}s | Rows: {len(gdf):,}')

speedup = cpu_time / max(gpu_time, 0.001)
print(f'\n🚀 cuDF Speedup: {speedup:.1f}x!')

print('\n📊 Data Preview (Top 5 rows of 5.8M total):')
print(gdf.head().to_pandas().to_string(index=False))

## Step 4: Convert to Parquet (MLOps Best Practice)

TSV is slow. Parquet is binary and 10x faster. We'll save the KG as Parquet for Phase 2.

In [ ]:
PROC_DIR = Path('data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_PATH = PROC_DIR / 'drkg.parquet'

print("📦 Optimizing: Saving as binary Parquet file...")
gdf.to_parquet(PARQUET_PATH)
print(f"✅ Saved to {PARQUET_PATH} ({PARQUET_PATH.stat().st_size / 1024**2:.1f} MB)")

## Step 5: Save To Google Drive

This ensures you don't have to download 5.8M rows every time you open Colab.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
SAVE_PATH = Path('/content/drive/MyDrive/BioKG_Project/data')
SAVE_PATH.mkdir(parents=True, exist_ok=True)

print("💾 Archiving to Google Drive...")
shutil.copy2(PARQUET_PATH, SAVE_PATH / 'drkg.parquet')
print(f"✅ Done! Your data is safe in Drive at: {SAVE_PATH}")